# 05 — Money: LTV, Profit Curve, Optimizer & Sensitivity
**RetainIQ India (Days 9-11).** The decision chain: probabilities → rupees → a budget-constrained contact list → stress tests.

> All monetary figures are **(simulation-based estimates)**. The dataset has ARPU and tenure but *no* cost, margin, offer or acceptance data — those are **declared assumptions** in `src/economics.py`.

In [1]:
import sys, pandas as pd, json
sys.path.insert(0,'../src')
from economics import ECON
print(json.dumps(ECON.as_dict(), indent=2))

{
  "gross_margin": 0.6,
  "monthly_discount_rate": 0.008,
  "horizon_months": 24,
  "offer_cost_inr": 120.0,
  "acceptance_rate": 0.35,
  "budget_inr": 100000.0
}


## 1. LTV — survival-weighted, not naive
`LTV = Σ ARPU × margin × S_contract(t) / (1+r)^t`. Using the Day-5 Kaplan-Meier curves means a month-to-month customer's future months are discounted by their *real* probability of still being there.

In [2]:
import ltv
vals = ltv.build()
vals.groupby('contract_type').agg(n=('customer_id','size'), mean_ltv=('ltv_inr','mean'),
    mean_churn=('churn_proba','mean'), mean_value_at_risk=('value_at_risk_inr','mean')).round(2)

05:59:07 | INFO    | model | leakage-free split: train=5634 test=1409 (stratified, seed=42)


05:59:07 | INFO    | model | Logistic (calibrated): {'roc_auc': 0.8441, 'pr_auc': 0.6595, 'brier': 0.1367, 'accuracy': 0.7999, 'precision': 0.6554, 'recall': 0.5187, 'f1': 0.5791}


05:59:07 | INFO    | model | Brier raw=0.1367 -> calibrated=0.1367


05:59:08 | INFO    | model | RandomForest (challenger): {'roc_auc': 0.846, 'pr_auc': 0.6705, 'brier': 0.1357, 'accuracy': 0.7999, 'precision': 0.6523, 'recall': 0.5267, 'f1': 0.5828}


05:59:08 | INFO    | model | saved primary model -> models/churn_model.pkl


...

05:59:09 | INFO    | model | mean LTV ₹708 | median ₹714 | total portfolio ₹0.5 Cr


05:59:09 | INFO    | model | total value at risk = ₹13.6 L  (simulation-based)


05:59:09 | INFO    | model | customers with positive expected_net at ₹120 offer: 1730 / 7043


05:59:09 | INFO    | model | LTV by contract:
                   n  mean_ltv  mean_churn  mean_var
contract_type                                       
Month-to-month  3875    619.88        0.43    291.46
One year        1473    841.56        0.11    116.92
Two year        1695    793.34        0.03     33.12


,n,mean_ltv,mean_churn,mean_value_at_risk
contract_type,,,,
Month-to-month,3875,619.88,0.43,291.46
One year,1473,841.56,0.11,116.92
Two year,1695,793.34,0.03,33.12


## 2. The rupee profit curve
For each contact threshold: benefit − cost. This exposes the **cost-sensitive threshold**.

In [3]:
import profit_curve as PC
pc = PC.run()
print('contact-everyone net : Rs', pc['contact_everyone']['net_retained_inr'])
print('profit-max threshold :', pc['optimal_threshold'])
print('  -> contact', pc['optimal']['n_contacted'], '| net Rs', pc['optimal']['net_retained_inr'])
print('per-customer +EV ceiling: Rs', pc['theoretical_ceiling_inr'])

05:59:09 | INFO    | model | Contact-everyone baseline: n=7043 net=₹-369938


05:59:09 | INFO    | model | PROFIT-MAXIMISING threshold = 0.50 | contact 1529 | net ₹74527


05:59:09 | INFO    | model | Theoretical ceiling (contact every +EV customer): ₹89549


05:59:09 | INFO    | model | Uplift vs contact-everyone: +120.1%  (simulation-based)


05:59:09 | INFO    | model | At the naive 0.50 cut: contact 1529 | net ₹74527 (leaves ₹0 on the table)


05:59:09 | INFO    | model | saved reports/profit_curve.json + reports/figures/profit_curve.png


contact-everyone net : Rs -369938.06
profit-max threshold : 0.5
  -> contact 1529 | net Rs 74527.17
per-customer +EV ceiling: Rs 89549.15


![profit curve](../reports/figures/profit_curve.png)

**Key insight.** The profit-max *global* threshold lands near 0.50 only by coincidence of these assumptions. Break-even churn probability actually varies **0.22 → 1.96** across customers because it depends on LTV. So **no single threshold is optimal** — which is exactly why we need an optimizer, not a cut-off.

## 3. The budget-constrained optimizer  ★ the thesis
0/1 knapsack: maximise Σ(benefit − cost) subject to Σcost ≤ budget. Solved greedily by ROI-per-rupee, verified against a DP bound.

In [4]:
import optimizer as O
res = O.run()
pd.DataFrame(res['results']).T[['n_contacted','spend_inr','net_retained_inr']]

05:59:09 | INFO    | model | Offer assignment (7,043 customers):


05:59:09 | INFO    | model | 
                   customers  avg_churn  avg_ltv  avg_net  total_cost
offer                                                                
Bill discount            147       0.70   935.47    73.42     32340.0
Courtesy call           1355       0.47   887.17    20.23     54200.0
Protection bundle        636       0.69   758.13    63.34     76320.0


05:59:09 | INFO    | model | eligible (+EV under best offer): 2138 / 7043


05:59:09 | INFO    | model | optimizer_roi        n= 1349 spend=₹   100000 net=₹     63133


05:59:09 | INFO    | model | rank_by_probability  n=  843 spend=₹    99980 net=₹     50934


...

05:59:09 | INFO    | model | uplift vs contact_everyone       +205.2%  (simulation-based)


05:59:09 | INFO    | model | DP knapsack bound ₹63133 vs greedy ₹63133 -> gap 0.000%


05:59:09 | INFO    | model | selected offer mix:
                     n    spend      net
offer                                   
Bill discount       50  11000.0   4835.0
Courtesy call      836  33440.0  23688.0
Protection bundle  463  55560.0  34610.0


05:59:09 | INFO    | model | saved reports/optimizer_result.json + reports/contact_list.csv


,n_contacted,spend_inr,net_retained_inr
optimizer_roi,1349.0,100000.0,63132.68
rank_by_probability,843.0,99980.0,50934.34
random,1993.0,99980.0,-18350.88
contact_everyone,7043.0,359060.0,-60009.06


In [5]:
print('budget utilisation :', res['budget_utilisation_pct'], '%')
print('greedy vs DP gap   :', res['greedy_gap_pct'], '%')
print('uplift vs baselines:', json.dumps(res['uplift_pct'], indent=2))

budget utilisation : 100.0 %
greedy vs DP gap   : 0.0 %
uplift vs baselines: {
  "rank_by_probability": 23.9,
  "random": 444.0,
  "contact_everyone": 205.2
}


**Headline `(simulation-based)`:** under a binding ₹1,00,000 budget the optimizer retains **₹63,133 net — +23.9% vs ranking by churn probability alone**, and +205% vs contacting everyone (which *loses* ₹60,009). Greedy matches the DP optimum exactly.

## 4. Sensitivity — which assumption breaks the recommendation first?

In [6]:
import sensitivity as SN
sn = SN.run()
print('break-even acceptance scale     :', sn['break_even_acceptance_scale'])
print('=> protection-bundle acceptance  :', sn['break_even_protection_acceptance_pct'], '% (baseline assumes 35%)')
pd.DataFrame(sn['one_way']['offer_cost_multiplier'])

05:59:10 | INFO    | model | Offer assignment (7,043 customers):


05:59:10 | INFO    | model | 
                   customers  avg_churn  avg_ltv  avg_net  total_cost
offer                                                                
Bill discount            147       0.70   935.47    73.42     32340.0
Courtesy call           1355       0.47   887.17    20.23     54200.0
Protection bundle        636       0.69   758.13    63.34     76320.0


05:59:10 | INFO    | model | eligible (+EV under best offer): 2138 / 7043


05:59:10 | INFO    | model | baseline net retained = ₹63133


05:59:10 | INFO    | model | Offer assignment (7,043 customers):


...

05:59:20 | INFO    | model | two-way grid (net retained ₹):
          cost x0.5  cost x1.0  cost x1.5  cost x2.0  cost x3.0
acc x0.4    18617.0      214.0        0.0        0.0        0.0
acc x0.7    86131.0    21943.0     2489.0        0.0        0.0
acc x1.0   143871.0    63133.0    26433.0     6039.0        0.0
acc x1.3   201120.0    99643.0    57040.0    29955.0     2132.0
acc x1.6   269323.0   130865.0    82231.0    54798.0    14684.0


05:59:20 | INFO    | model | most influential assumption: offer_cost_multiplier


05:59:20 | INFO    | model | saved reports/sensitivity.json + tornado_sensitivity.png


break-even acceptance scale     : 0.35
=> protection-bundle acceptance  : 12.2 % (baseline assumes 35%)


,offer_cost_multiplier,net_retained_inr
0,0.50,143871.0
1,0.75,94486.0
2,1.00,63133.0
3,1.50,26433.0
4,2.00,6039.0
5,3.00,0.0


![tornado](../reports/figures/tornado_sensitivity.png)

**Offer cost is the most influential assumption.** The campaign survives down to ~12.2% acceptance — a ~3× cushion under our 35% assumption.

## 5. What-if scenarios

In [7]:
import simulator as S
sc = S.run()
pd.DataFrame(sc['scenarios']).T[['n_contacted','spend_inr','net_retained_inr']]

05:59:20 | INFO    | model | Offer assignment (7,043 customers):


05:59:20 | INFO    | model | 
Empty DataFrame
Columns: [customers, avg_churn, avg_ltv, avg_net, total_cost]
Index: []


05:59:20 | INFO    | model | eligible (+EV under best offer): 0 / 7043


05:59:20 | INFO    | model | pessimistic  net=₹        0 | contacted=    0 | util=  0.0% | mix={}


05:59:20 | INFO    | model | Offer assignment (7,043 customers):


...

05:59:21 | INFO    | model | eligible (+EV under best offer): 2138 / 7043


05:59:21 | INFO    | model | budget response:
 budget_inr  net_retained_inr  n_contacted  budget_utilisation_pct
      10000          10917.36          184                    99.2
      25000          24074.96          405                    99.7
      50000          40601.64          767                    99.8
      75000          53091.97         1062                    99.9
     100000          63132.68         1349                   100.0
     150000          77104.96         1845                   100.0
     200000          78490.00         2138                    81.4
     300000          78490.00         2138                    54.3


05:59:21 | INFO    | model | budget saturates near ₹200,000 — beyond this, extra budget buys nothing (no +EV customers left)


05:59:21 | INFO    | model | saved reports/scenarios.json


,n_contacted,spend_inr,net_retained_inr
pessimistic,0,0.0,0.0
base,1349,100000.0,63132.68
optimistic,741,99990.0,158924.89


**The pessimistic scenario contacts nobody.** Under weak acceptance, expensive offers and thin margin, *no* customer is positive-EV — so the system correctly recommends **not running the campaign**. A decision tool that knows when not to spend is more trustworthy than one that always does.

**Budget saturates near ₹2,00,000** — beyond that, extra budget buys nothing because no +EV customers remain.

*Next: `docs/AB_experiment_design.md` — how we'd replace the acceptance **assumption** with a **measurement**.*